# Challenge

# Laboratorio pomeridiano — Classificazione di immagini con reti neurali

### Fondazione iFAB · Corso IFTS — Machine Learning e Reti Neurali

In questo laboratorio costruirete, addestrerete e **ottimizzerete** una rete neurale per riconoscere immagini, lavorando in **5 fasi progressive**.

> **Nessuna installazione, nessun download manuale.** Usiamo dataset già pronti o scaricati automaticamente la prima volta. Tutto gira su **Google Colab** dal browser, anche senza un PC potente.

---

### Cosa imparerete oggi
1. Caricare ed esplorare un dataset di immagini reali
2. **Diagnosticare l'overfitting** guardando le curve di apprendimento
3. Migliorare il modello con una **CNN**
4. **Sfida finale a fasi:** ottimizzare per superare il **90% di accuracy** senza overfitting

### Il dataset
I dataset disponibili in questo laboratorio sono:
- **patatine**: immagini reali di controllo qualità
- **fashion**: capi di abbigliamento 28x28

Il confronto tra questi casi vi permette di vedere come cambia il comportamento del modello al variare dei dati.

---

### Come lavorare
Lavorate in **gruppi di 2-3 persone**. Discutete ogni scelta prima di scriverla nel codice: *perché* questo numero di neuroni? *perché* questa modifica? Alla fine ogni gruppo racconterà cosa ha provato.

## Regole
- NON modificate l'architettura della rete
- Potete modificare SOLO gli iperparametri indicati
- Vince chi ottiene la migliore accuracy sul TEST SET senza overfitting

> **Nota**: Questa è la versione locale di backup. I dati vengono caricati dalla cartella `data/`. Selezionate il dataset nella cella di caricamento.

In [ ]:
# Importazione librerie (NON MODIFICARE)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.utils import shuffle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
import time

# Riproducibilità
np.random.seed(1234)
tf.random.set_seed(1234)

print("Librerie caricate correttamente!")
print(f"TensorFlow version: {tf.__version__}")

: 

In [ ]:
# ============================================================
#  SCEGLI QUI IL DATASET:
#    "patatine" -> controllo qualita chips (si scarica dal Drive condiviso)
#    "fashion"  -> capi di abbigliamento
# ============================================================

DATASET = "fashion"

LINK_DRIVE_CHIPS = "https://drive.google.com/drive/folders/1wmufMDUDOuNIj8hD0CT-ZkRFvtJ6ge4Q"

import os

def trova_file(base_dir, nome_file):
    for radice, _, files in os.walk(base_dir):
        if nome_file in files:
            return os.path.join(radice, nome_file)
    raise FileNotFoundError(f"Non trovo {nome_file}.")

def prepara_dataset(nome_dataset):
    if nome_dataset == "patatine":
        import importlib.util, subprocess, sys
        if importlib.util.find_spec("gdown") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown", "-q"])
        import gdown

        data_dir = "data_chips"
        if not os.path.exists(data_dir):
            gdown.download_folder(LINK_DRIVE_CHIPS, output=data_dir, quiet=False, use_cookies=False)

        # Elimina i _500 subito dopo il download, prima di caricare in RAM
        for f in ['chips_cnn_X_500.npy', 'chips_cnn_y_500.npy',
                  'chips_cnn_X_test_500.npy', 'chips_cnn_y_test_500.npy']:
            try:
                os.remove(trova_file(data_dir, f))
                print(f"    Rimosso {f}")
            except FileNotFoundError:
                pass

        X      = np.load(trova_file(data_dir, 'chips_cnn_X_100.npy'))
        y      = np.load(trova_file(data_dir, 'chips_cnn_y_100.npy'))
        X_test = np.load(trova_file(data_dir, 'chips_cnn_X_test_100.npy'))
        y_test = np.load(trova_file(data_dir, 'chips_cnn_y_test_100.npy'))
        X, y = shuffle(X, y, random_state=1234)
        X_test, y_test = shuffle(X_test, y_test, random_state=1234)
        if X.ndim == 3:      X      = X[..., np.newaxis]
        if X_test.ndim == 3: X_test = X_test[..., np.newaxis]
        nomi_classi    = ['Difettose (bruciate)', 'Buone']
        target_accuracy  = 0.97
        gia_normalizzato = True

    elif nome_dataset == "fashion":
        (X, y), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
        X = X[..., np.newaxis]; X_test = X_test[..., np.newaxis]
        X, y = shuffle(X, y, random_state=1234)
        X_test, y_test = shuffle(X_test, y_test, random_state=1234)
        nomi_classi = ['T-shirt/top','Pantaloni','Pullover','Vestito','Cappotto',
                       'Sandalo','Camicia','Scarpa ginnastica','Borsa','Stivaletto']
        target_accuracy = 0.90; gia_normalizzato = False

    else:
        raise ValueError("DATASET deve essere 'patatine' o 'fashion'")

    return X, y, X_test, y_test, nomi_classi, target_accuracy, gia_normalizzato

X, y, X_test, y_test, nomi_classi, TARGET_ACCURACY, GIA_NORMALIZZATO = prepara_dataset(DATASET)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)

# Normalizzazione: fashion arriva come uint8 (0-255)
if not GIA_NORMALIZZATO:
    X_train = X_train.astype('float32') / 255.0
    X_val   = X_val.astype('float32')   / 255.0
    X_test  = X_test.astype('float32')  / 255.0

n_classi = len(nomi_classi)
canali = X.shape[-1] if X.ndim == 4 else 1
lato = X.shape[1]

print(f'✅ Dati caricati!')
print(f'   Dataset scelto : {DATASET}')
print(f'   Training       : {X_train.shape[0]} immagini')
print(f'   Validazione    : {X_val.shape[0]} immagini')
print(f'   Test           : {X_test.shape[0]} immagini')
print(f'   Classi         : {n_classi}')
print(f'   Obiettivo acc  : {TARGET_ACCURACY*100:.0f}%')

In [ ]:
# Dataset _100: immagini già a bassa risoluzione, nessun resize necessario
print(f'✅ Immagini caricate dal file _100')
print(f'   Shape X_train : {X_train.shape}')
print(f'   RAM stimata   : ~{X_train.nbytes // 1_000_000} MB')

In [ ]:
# Visualizza alcuni esempi (NON MODIFICARE)
if n_classi == 2:
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    idx_class0 = np.where(y == 0)[0][:4]
    idx_class1 = np.where(y == 1)[0][:4]
    for i in range(4):
        img0 = np.squeeze(X[idx_class0[i]])
        img1 = np.squeeze(X[idx_class1[i]])
        axes[0, i].imshow(img0, cmap='gray' if canali == 1 else None)
        axes[0, i].set_title(f'Classe 0: {nomi_classi[0]}', color='red')
        axes[0, i].axis('off')
        axes[1, i].imshow(img1, cmap='gray' if canali == 1 else None)
        axes[1, i].set_title(f'Classe 1: {nomi_classi[1]}', color='green')
        axes[1, i].axis('off')
else:
    n_mostra = min(n_classi, 10)
    fig, axes = plt.subplots(n_mostra, 4, figsize=(12, 3 * n_mostra))
    for c in range(n_mostra):
        idx_c = np.where(y == c)[0][:4]
        for i in range(min(4, len(idx_c))):
            ax = axes[c, i] if n_mostra > 1 else axes[i]
            ax.imshow(np.squeeze(X[idx_c[i]]), cmap='gray' if canali == 1 else None)
            ax.set_title(nomi_classi[c], fontsize=9)
            ax.axis('off')

plt.tight_layout()
plt.show()

---
# ZONA MODIFICABILE — IPERPARAMETRI

**Modificate SOLO questa cella!** Sperimentate con diversi valori per trovare la combinazione migliore.

---

In [ ]:
#╔══════════════════════════════════════════════════════════════════╗
#║                    MODIFICATE QUI!                              ║
#╠══════════════════════════════════════════════════════════════════╣
#║  Cambiate questi valori e osservate come cambiano i risultati   ║
#╚══════════════════════════════════════════════════════════════════╝

# Numero di epoche (quante volte la rete vede tutti i dati)
# Range consigliato: 5 - 100
EPOCHS = 30

# Batch size (quante immagini alla volta)
# Range consigliato: 32, 64, 128, 256
BATCH_SIZE = 128

# Learning rate (velocità di apprendimento)
# Range consigliato: 0.00001 - 0.01
LEARNING_RATE = 0.0001

# Dropout rate (percentuale di neuroni "spenti" per regolarizzare)
# Range consigliato: 0.2 - 0.7
DROPOUT_RATE = 0.5

# Regolarizzazione L2 (penalizza pesi troppo grandi)
# Range consigliato: 0.0001 - 0.01
L2_REGULARIZATION = 0.0015

#══════════════════════════════════════════════════════════════════

print("Configurazione attuale:")
print(f"   Epoche: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Dropout: {DROPOUT_RATE}")
print(f"   Regolarizzazione L2: {L2_REGULARIZATION}")

---
# Costruzione e Training della Rete

---

In [ ]:
# Preparazione dataset (NON MODIFICARE)
# NOTA: .repeat(2) raddoppia i dati in ogni epoca (data augmentation implicita).
# Con EPOCHS=30 la rete vedrà i dati effettivamente 60 volte.
# Questo aiuta su dataset piccoli ma rende le curve di training meno intuitive.
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.repeat(2).shuffle(2 * BATCH_SIZE).batch(BATCH_SIZE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.repeat(2).shuffle(2 * BATCH_SIZE).batch(BATCH_SIZE)

print("Dataset preparati!")
print(f"   Ogni epoca itera su 2x i dati (repeat=2) → {EPOCHS} epoche = {EPOCHS*2} passaggi effettivi sui dati")

In [ ]:
# ============================================================
# Costruzione modello CNN — costruite la rete voi!
# ============================================================
reg = tf.keras.regularizers.l2(l2=L2_REGULARIZATION)

model = Sequential([
    layers.Input(shape=X_train.shape[1:]),

    # ============================================================
    # Costruite qui la vostra rete.
    #
    # Strati disponibili:
    #   layers.Conv2D(filtri, 3, padding='same', activation='relu'), es. layers.Conv2D(filters=32, kernel_size=(3,3), activation='relu'),
    #   layers.Conv2D(filtri, 3, padding='same', activation='relu', kernel_regularizer=reg),
    #   layers.MaxPooling2D(),
    #   layers.BatchNormalization(),
    #   layers.Dropout(percentuale),          # es. layers.Dropout(0.3)
    #   layers.Flatten(),                     # obbligatorio prima dei Dense
    #   layers.GlobalAveragePooling2D(),      # alternativa a Flatten
    #   layers.Dense(neuroni, activation='relu'),      # es. layers.Dense(128, activation='relu'),
    #   layers.Dense(neuroni, activation='relu', kernel_regularizer=reg),
    #
    # Regole:
    #   - Servono almeno un Conv2D + MaxPooling2D e un Flatten/GlobalAveragePooling2D
    #   - Dopo Flatten/GlobalAveragePooling2D mettete solo layer Dense
    #   - Non toccate la riga Input sopra e la riga Dense finale sotto
    # ============================================================

    layers.Dense(n_classi, activation='softmax')
])

opt = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(
    optimizer=opt,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

print("\nArchitettura della rete:")
model.summary()

In [ ]:
# Training (NON MODIFICARE)
print("Inizio training...\n")
start_time = time.time()

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    verbose=1
)

training_time = time.time() - start_time
print(f"\n✅ Training completato in {training_time:.1f} secondi")

In [ ]:
# Grafici risultati (NON MODIFICARE)
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(EPOCHS)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, 'b-', label='Accuracy Training', linewidth=2)
plt.plot(epochs_range, val_acc, 'r-', label='Accuracy Validazione', linewidth=2)
plt.axhline(y=TARGET_ACCURACY, color='g', linestyle='--', label=f'Obiettivo {TARGET_ACCURACY*100:.0f}%')
plt.legend(loc='lower right')
plt.title('Accuracy', fontsize=14)
plt.xlabel('Epoche')
plt.ylabel('Accuracy')
plt.ylim([max(0.0, min(acc + val_acc) - 0.05), 1.0])
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, 'b-', label='Loss Training', linewidth=2)
plt.plot(epochs_range, val_loss, 'r-', label='Loss Validazione', linewidth=2)
plt.legend(loc='upper right')
plt.title('Loss', fontsize=14)
plt.xlabel('Epoche')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Analisi overfitting
final_train_acc = acc[-1]
final_val_acc = val_acc[-1]
gap = final_train_acc - final_val_acc

print(f"\nAnalisi finale del training:")
print(f"   Accuracy Training finale: {final_train_acc:.4f} ({final_train_acc*100:.2f}%)")
print(f"   Accuracy Validazione finale: {final_val_acc:.4f} ({final_val_acc*100:.2f}%)")
print(f"   Gap (Train - Val): {gap:.4f}")

if gap > 0.05:
    print("   ATTENZIONE: Possibile overfitting! Il gap è troppo alto.")
elif final_val_acc < TARGET_ACCURACY:
    print(f"   Non hai ancora raggiunto il {TARGET_ACCURACY*100:.0f}%! Continua a sperimentare.")
else:
    print("   Ottimo! Sembra un buon training.")

---
# RISULTATO FINALE SUL TEST SET

Questo è il valore che conta per la classifica!

---

In [ ]:
# Valutazione sul TEST SET (NON MODIFICARE)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

# Ricalcolo gap per sicurezza (nel caso le celle vengano eseguite fuori ordine)
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
gap = acc[-1] - val_acc[-1]

print("="*50)
print("RISULTATO FINALE")
print("="*50)
print(f"\n   TEST ACCURACY: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   TEST LOSS: {test_loss:.4f}")
print("\n" + "="*50)

if test_acc >= TARGET_ACCURACY and gap <= 0.05:
    print("\nCONGRATULAZIONI! Hai raggiunto l'obiettivo!")
    print(f"   Accuracy >= {TARGET_ACCURACY*100:.0f}% e nessun overfitting significativo!")
elif test_acc >= TARGET_ACCURACY:
    print(f"\nHai raggiunto il {TARGET_ACCURACY*100:.0f}%, ma attenzione all'overfitting!")
else:
    print(f"\nContinua a provare! Ti manca {(TARGET_ACCURACY - test_acc)*100:.2f}% per raggiungere l'obiettivo.")

In [ ]:
# Matrice di Confusione (NON MODIFICARE)
y_pred = model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)

conf_matrix = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(max(8, n_classi), max(6, n_classi)))
plt.imshow(conf_matrix, interpolation='nearest', cmap='Blues')
plt.title('Matrice di Confusione', fontsize=14)
plt.colorbar()

classes = nomi_classi
tick_marks = np.arange(n_classi)
plt.xticks(tick_marks, classes, rotation=45, ha='right')
plt.yticks(tick_marks, classes)
plt.xlabel('Etichetta Predetta')
plt.ylabel('Etichetta Vera')

thresh = conf_matrix.max() / 2.
for i in range(n_classi):
    for j in range(n_classi):
        plt.text(j, i, format(conf_matrix[i, j], 'd'),
                 ha="center", va="center", fontsize=max(6, 16 - n_classi),
                 color="white" if conf_matrix[i, j] > thresh else "black")

plt.tight_layout()
plt.show()

print(f"\nErrori di classificazione:")
if n_classi == 2:
    print(f"   {nomi_classi[0]} classificate come {nomi_classi[1]} (Falsi Negativi): {conf_matrix[0,1]}")
    print(f"   {nomi_classi[1]} classificate come {nomi_classi[0]} (Falsi Positivi): {conf_matrix[1,0]}")
else:
    print("   Dataset multicategoria: usa la matrice per individuare le confusioni principali.")